In [6]:
import os
import joblib
import mlflow
import mlflow.sklearn
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE

In [ ]:
df = pd.read_csv("Loan_Data.csv", sep=";", skiprows=1)

In [8]:
print(df.shape)
print(df.columns)
print(df["default"].value_counts())
print(df["default"].value_counts(normalize=True))

(10000, 8)
Index(['customer_id', 'credit_lines_outstanding', 'loan_amt_outstanding',
       'total_debt_outstanding', 'income', 'years_employed', 'fico_score',
       'default'],
      dtype='object')
default
0    8149
1    1851
Name: count, dtype: int64
default
0    0.8149
1    0.1851
Name: proportion, dtype: float64


In [9]:
target_col = "default"

X = df.drop(columns=[target_col, "customer_id"])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("\nRépartition y_train avant SMOTE :")
print(y_train.value_counts())

X_train : (7000, 6)
X_test  : (3000, 6)

Répartition y_train avant SMOTE :
default
0    5704
1    1296
Name: count, dtype: int64


In [10]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
    }
    return metrics

In [11]:
os.makedirs("../models", exist_ok=True)

In [12]:
def run_experiment(experiment_name, run_name, model, params, use_smote=False):
    mlflow.set_experiment(experiment_name)

    with mlflow.start_run(run_name=run_name):
        # données d'entraînement
        X_train_used = X_train.copy()
        y_train_used = y_train.copy()

        # appliquer SMOTE uniquement sur train
        if use_smote:
            smote = SMOTE(random_state=42)
            X_train_used, y_train_used = smote.fit_resample(X_train_used, y_train_used)

        # entraînement
        model.fit(X_train_used, y_train_used)

        # évaluation sur le vrai jeu de test
        metrics = evaluate_model(model, X_test, y_test)

        # log paramètres
        all_params = params.copy()
        all_params["use_smote"] = use_smote
        mlflow.log_params(all_params)

        # log métriques
        mlflow.log_metrics(metrics)

        # sauvegarde locale
        model_dir = "../models" if os.path.basename(os.getcwd()) == "notebooks" else "models"
        model_path = f"{model_dir}/{run_name}.joblib"
        joblib.dump(model, model_path)

        # log modèle mlflow
        mlflow.sklearn.log_model(model, artifact_path="model")
        mlflow.log_artifact(model_path)

        print(f"Experiment: {experiment_name}")
        print(f"Run: {run_name}")
        print(f"SMOTE: {use_smote}")
        print(metrics)
        print("-" * 50)

        return {
            "experiment": experiment_name,
            "run_name": run_name,
            "use_smote": use_smote,
            **metrics
        }

In [13]:
results = []

results.append(
    run_experiment(
        experiment_name="Logistic_Regression",
        run_name="logreg_baseline_no_smote",
        model=LogisticRegression(max_iter=1000, random_state=42),
        params={
            "model_type": "LogisticRegression",
            "max_iter": 1000,
            "random_state": 42
        },
        use_smote=False
    )
)

results.append(
    run_experiment(
        experiment_name="Logistic_Regression",
        run_name="logreg_baseline_smote",
        model=LogisticRegression(max_iter=1000, random_state=42),
        params={
            "model_type": "LogisticRegression",
            "max_iter": 1000,
            "random_state": 42
        },
        use_smote=True
    )
)

results.append(
    run_experiment(
        experiment_name="Logistic_Regression",
        run_name="logreg_C_0_5_no_smote",
        model=LogisticRegression(C=0.5, max_iter=1000, random_state=42),
        params={
            "model_type": "LogisticRegression",
            "C": 0.5,
            "max_iter": 1000,
            "random_state": 42
        },
        use_smote=False
    )
)

results.append(
    run_experiment(
        experiment_name="Logistic_Regression",
        run_name="logreg_C_0_5_smote",
        model=LogisticRegression(C=0.5, max_iter=1000, random_state=42),
        params={
            "model_type": "LogisticRegression",
            "C": 0.5,
            "max_iter": 1000,
            "random_state": 42
        },
        use_smote=True
    )
)

2026/04/02 17:50:57 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/02 17:50:57 INFO mlflow.store.db.utils: Updating database tables
2026/04/02 17:50:58 INFO mlflow.tracking.fluent: Experiment with name 'Logistic_Regression' does not exist. Creating a new experiment.
2026/04/02 17:50:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/02 17:50:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/02 17:51:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Experiment: Logistic_Regression
Run: logreg_baseline_no_smote
SMOTE: False
{'accuracy': 0.9983333333333333, 'recall': 0.9963963963963964, 'f1_score': 0.9954995499549955, 'roc_auc': np.float64(0.9999889460012159)}
--------------------------------------------------


2026/04/02 17:51:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/02 17:51:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Experiment: Logistic_Regression
Run: logreg_baseline_smote
SMOTE: True
{'accuracy': 0.995, 'recall': 1.0, 'f1_score': 0.9866666666666667, 'roc_auc': np.float64(0.9999322021407911)}
--------------------------------------------------


2026/04/02 17:51:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/02 17:51:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/02 17:51:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Experiment: Logistic_Regression
Run: logreg_C_0_5_no_smote
SMOTE: False
{'accuracy': 0.9973333333333333, 'recall': 0.9945945945945946, 'f1_score': 0.9928057553956835, 'roc_auc': np.float64(0.9999823136019454)}
--------------------------------------------------
Experiment: Logistic_Regression
Run: logreg_C_0_5_smote
SMOTE: True
{'accuracy': 0.9943333333333333, 'recall': 1.0, 'f1_score': 0.9849157054125999, 'roc_auc': np.float64(0.9999130418762321)}
--------------------------------------------------


In [14]:
results.append(
    run_experiment(
        experiment_name="Decision_Tree",
        run_name="dt_baseline_no_smote",
        model=DecisionTreeClassifier(random_state=42),
        params={
            "model_type": "DecisionTreeClassifier",
            "random_state": 42
        },
        use_smote=False
    )
)

results.append(
    run_experiment(
        experiment_name="Decision_Tree",
        run_name="dt_baseline_smote",
        model=DecisionTreeClassifier(random_state=42),
        params={
            "model_type": "DecisionTreeClassifier",
            "random_state": 42
        },
        use_smote=True
    )
)

results.append(
    run_experiment(
        experiment_name="Decision_Tree",
        run_name="dt_max_depth_5_no_smote",
        model=DecisionTreeClassifier(max_depth=5, random_state=42),
        params={
            "model_type": "DecisionTreeClassifier",
            "max_depth": 5,
            "random_state": 42
        },
        use_smote=False
    )
)

results.append(
    run_experiment(
        experiment_name="Decision_Tree",
        run_name="dt_max_depth_5_smote",
        model=DecisionTreeClassifier(max_depth=5, random_state=42),
        params={
            "model_type": "DecisionTreeClassifier",
            "max_depth": 5,
            "random_state": 42
        },
        use_smote=True
    )
)

2026/04/02 17:51:10 INFO mlflow.tracking.fluent: Experiment with name 'Decision_Tree' does not exist. Creating a new experiment.
2026/04/02 17:51:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/02 17:51:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/02 17:51:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/02 17:51:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserializ

Experiment: Decision_Tree
Run: dt_baseline_no_smote
SMOTE: False
{'accuracy': 0.9953333333333333, 'recall': 0.9873873873873874, 'f1_score': 0.9873873873873874, 'roc_auc': np.float64(0.992262200851158)}
--------------------------------------------------


2026/04/02 17:51:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/02 17:51:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Experiment: Decision_Tree
Run: dt_baseline_smote
SMOTE: True
{'accuracy': 0.989, 'recall': 0.9927927927927928, 'f1_score': 0.9709251101321585, 'roc_auc': np.float64(0.9904659260487481)}
--------------------------------------------------


2026/04/02 17:51:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/02 17:51:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Experiment: Decision_Tree
Run: dt_max_depth_5_no_smote
SMOTE: False
{'accuracy': 0.9966666666666667, 'recall': 0.9891891891891892, 'f1_score': 0.9909747292418772, 'roc_auc': np.float64(0.9993879769339892)}
--------------------------------------------------
Experiment: Decision_Tree
Run: dt_max_depth_5_smote
SMOTE: True
{'accuracy': 0.993, 'recall': 0.9927927927927928, 'f1_score': 0.981300089047195, 'roc_auc': np.float64(0.998826802262385)}
--------------------------------------------------


In [15]:
results.append(
    run_experiment(
        experiment_name="Random_Forest",
        run_name="rf_100_no_smote",
        model=RandomForestClassifier(n_estimators=100, random_state=42),
        params={
            "model_type": "RandomForestClassifier",
            "n_estimators": 100,
            "random_state": 42
        },
        use_smote=False
    )
)

results.append(
    run_experiment(
        experiment_name="Random_Forest",
        run_name="rf_100_smote",
        model=RandomForestClassifier(n_estimators=100, random_state=42),
        params={
            "model_type": "RandomForestClassifier",
            "n_estimators": 100,
            "random_state": 42
        },
        use_smote=True
    )
)

results.append(
    run_experiment(
        experiment_name="Random_Forest",
        run_name="rf_200_no_smote",
        model=RandomForestClassifier(n_estimators=200, random_state=42),
        params={
            "model_type": "RandomForestClassifier",
            "n_estimators": 200,
            "random_state": 42
        },
        use_smote=False
    )
)

results.append(
    run_experiment(
        experiment_name="Random_Forest",
        run_name="rf_200_smote",
        model=RandomForestClassifier(n_estimators=200, random_state=42),
        params={
            "model_type": "RandomForestClassifier",
            "n_estimators": 200,
            "random_state": 42
        },
        use_smote=True
    )
)

2026/04/02 17:51:20 INFO mlflow.tracking.fluent: Experiment with name 'Random_Forest' does not exist. Creating a new experiment.
2026/04/02 17:51:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/02 17:51:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Experiment: Random_Forest
Run: rf_100_no_smote
SMOTE: False
{'accuracy': 0.9963333333333333, 'recall': 0.990990990990991, 'f1_score': 0.9900990099009901, 'roc_auc': np.float64(0.9998533502827981)}
--------------------------------------------------


2026/04/02 17:51:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/02 17:51:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Experiment: Random_Forest
Run: rf_100_smote
SMOTE: True
{'accuracy': 0.991, 'recall': 0.990990990990991, 'f1_score': 0.9760425909494232, 'roc_auc': np.float64(0.9996753809023747)}
--------------------------------------------------


2026/04/02 17:51:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/02 17:51:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Experiment: Random_Forest
Run: rf_200_no_smote
SMOTE: False
{'accuracy': 0.9966666666666667, 'recall': 0.990990990990991, 'f1_score': 0.990990990990991, 'roc_auc': np.float64(0.9998533502827982)}
--------------------------------------------------


2026/04/02 17:51:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/02 17:51:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Experiment: Random_Forest
Run: rf_200_smote
SMOTE: True
{'accuracy': 0.9916666666666667, 'recall': 0.9927927927927928, 'f1_score': 0.9778172138420586, 'roc_auc': np.float64(0.9996960150334383)}
--------------------------------------------------


In [16]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=["recall", "f1_score", "roc_auc"], ascending=False)
results_df

,experiment,run_name,use_smote,accuracy,recall,f1_score,roc_auc
1,Logistic_Regression,logreg_baseline_smote,True,0.995000,1.000000,0.986667,0.999932
3,Logistic_Regression,logreg_C_0_5_smote,True,0.994333,1.000000,0.984916,0.999913
0,Logistic_Regression,logreg_baseline_no_smote,False,0.998333,0.996396,0.995500,0.999989
2,Logistic_Regression,logreg_C_0_5_no_smote,False,0.997333,0.994595,0.992806,0.999982
7,Decision_Tree,dt_max_depth_5_smote,True,0.993000,0.992793,0.981300,0.998827
11,Random_Forest,rf_200_smote,True,0.991667,0.992793,0.977817,0.999696
5,Decision_Tree,dt_baseline_smote,True,0.989000,0.992793,0.970925,0.990466
10,Random_Forest,rf_200_no_smote,False,0.996667,0.990991,0.990991,0.999853
8,Random_Forest,rf_100_no_smote,False,0.996333,0.990991,0.990099,0.999853
9,Random_Forest,rf_100_smote,True,0.991000,0.990991,0.976043,0.999675


In [17]:
results_df[[
    "experiment", "run_name", "use_smote", "accuracy", "recall", "f1_score", "roc_auc"
]]

,experiment,run_name,use_smote,accuracy,recall,f1_score,roc_auc
1,Logistic_Regression,logreg_baseline_smote,True,0.995000,1.000000,0.986667,0.999932
3,Logistic_Regression,logreg_C_0_5_smote,True,0.994333,1.000000,0.984916,0.999913
0,Logistic_Regression,logreg_baseline_no_smote,False,0.998333,0.996396,0.995500,0.999989
2,Logistic_Regression,logreg_C_0_5_no_smote,False,0.997333,0.994595,0.992806,0.999982
7,Decision_Tree,dt_max_depth_5_smote,True,0.993000,0.992793,0.981300,0.998827
11,Random_Forest,rf_200_smote,True,0.991667,0.992793,0.977817,0.999696
5,Decision_Tree,dt_baseline_smote,True,0.989000,0.992793,0.970925,0.990466
10,Random_Forest,rf_200_no_smote,False,0.996667,0.990991,0.990991,0.999853
8,Random_Forest,rf_100_no_smote,False,0.996333,0.990991,0.990099,0.999853
9,Random_Forest,rf_100_smote,True,0.991000,0.990991,0.976043,0.999675
